# Superstore 销售利润分析报告

## 一、项目背景与目标

### 1.1 项目背景

    某零售公司（Superstore）主营办公家具、技术产品和办公用品，客户覆盖消费者、公司及家庭办公三大群体，销售网络遍布美国多个地区。近年来，公司整体销售额和利润保持增长，但管理层发现部分产品子类别（如 Tables、Bookcases）持续亏损，且整体折扣成本逐年上升。

核心痛点：

- 整体利润率约为12%，但不同产品、地区之间差异显著。

- 部分高折扣订单并未带来预期销量增长，反而侵蚀利润。

- 缺乏对“折扣–利润”关系的量化分析，促销策略依赖经验判断。

为此，通过对历史销售数据的深度分析，识别亏损根源，量化折扣影响，并制定科学的折扣策略，以提升整体盈利能力。

### 1.2 分析目标

本分析基于 2014–2017 年共 9,994 条订单记录，围绕以下四个层面展开：

| 分析层次 | 核心问题 | 预期产出 |
| :------: | :------: | :------: |
| 描述性分析	| 整体销售利润状况？哪些产品/地区/客户贡献主要利润？哪些在亏损？ | 利润分布仪表板、亏损产品清单 |
诊断性分析 | 为什么这些产品亏损？折扣是否是主因？不同客户群的折扣敏感度如何？ | 折扣与利润的量化关系（回归系数）、亏损订单的折扣特征 |
预测性分析 | 未来哪些订单可能亏损？哪些客户群风险最高？ | 亏损订单预测模型、高风险客户标签 |
规范性分析 | 折扣上限设为多少最优？如何调整促销策略？ | 折扣模拟器、可落地的业务建议（如设置20%折扣上限） |

### 1.3 分析价值

直接价值：通过量化折扣的边际效应，为定价和促销提供数据依据，预计可减少亏损订单约 30%–40%。

长期价值：建立客户价值分层（RFM）和亏损预测模型，支持精细化运营。

可复现性：所有分析代码、SQL脚本、Excel模拟器及Power BI仪表板均已开源，便于业务团队持续监控。

## 二、数据来源与字段说明

### 2.1 数据源

- 原始数据集：Kaggle 公开数据集 Sample Superstore

- 时间范围：2014-01-01 至 2017-12-31

- 记录数：9,994 条订单（去重后 9,977 条）

### 2.2 核心字段

| 字段名 | 格式 | 描述 |
|-----|------|--------|
| Order ID | 字符串 | 订单编号 |
| Order Date | 字符串 | 订单日期 |
| Customer ID | 字符串 | 客户编号 |
| Segment | 字符串 | 客户细分市场 |
| State | 字符串 | 州 |
| Region | 字符串 | 地区 |
| Product ID | 字符串 | 产品编号 |
| Category | 字符串 | 产品大类 |
| Sub-Category | 字符串 | 产品子类 |
| Sales | 浮点数 | 销售额 |
| Quantity | 整数   | 销售数量 |
| Discount | 浮点数 | 折扣比例 |
| Profit | 浮点数 | 利润 |

### 2.3 数据预处理

- 使用 Python (pandas) 读取数据，处理缺失值（无缺失）

- 删除重复行（无重复）

- 将 Order Date 转换为 datetime 类型，并提取年份、月份、季度

- 创建宽表 orders_wide，增加 is_loss（是否亏损）、discount_amount（折扣金额）、is_holiday_season（是否节假日）等特征

- 通过 SQL 导出客户 RFM 表，包含 recency、frequency、monetary、avg_discount 等字段

## 三、分析过程

### 3.1 整体数据概览

- 总销售额: 2,297,200.86
- 总利润: 286,397.02
- 整体利润率: 12.47%

公司整体处于盈利状态，但需进一步分析是否存在亏损产品或区域。

### 3.2 亏损产品与区域识别

#### 3.2.1 按子类别(Sub-Category)分析利润(仅展示亏损子类别)

| Sub-Category | 总利润 | 利润率 |
| ----- | ----- | ----- |
| Supplies | -1189.0995 | -2.547695 |
| Bookcases | -3472.5560 | -3.022768 |
| Tables | -17725.4811 | -8.564460 |

#### 3.2.2 按区域分析利润

各区域总利润排序：

| 区域     | 利润         |
|----------|--------------|
| West     | 108418.4489  |
| East     | 91522.7800   |
| South    | 46749.4303   |
| Central  | 39706.3625   |

中部地区（Central）利润最低，需重点关注。

### 3.3 折扣与利润关系分析

#### 3.3.1 散点图与折扣区间分析

![折扣与利润关系](discount_profit.png)

不同折扣区间的平均利润：

| 折扣区间 | 平均利润 |
|----------|----------|
| 0-10%    | 66.90   |
| 10-20%   | 71.56   |
| 20-30%   | 24.70   |
| 30-40%   | -50.24  |
| 40-50%   | -117.74  |
| 50%+     | -105.28   |

当折扣超过 30% 时，平均利润转为负值，且折扣越高亏损越严重。

#### 3.3.2 盈利订单 vs 亏损订单折扣对比

- 盈利订单平均折扣仅为 8%，而亏损订单平均折扣高达 48%。
- 亏损订单的折扣比盈利订单高出: 39.98%

### 3.4 时间趋势分析

按月汇总销售额和利润，绘制双轴折线图。结果显示：**第四季度（10-12月）销售额和利润均达到峰值，但折扣力度也较大，需警惕高折扣对利润的侵蚀。**

### 3.5 深度归因分析

#### 3.5.1 线性回归（归因分析）

以利润为因变量，折扣、销售额、产品类别、地区为自变量，建立多元线性回归模型。关键结果：

| 变量                        | 系数     | P值     |
|----------------------------|----------|---------|
| Discount                   | -236.82  | <0.001  |
| Sales                      | 0.182    | <0.001  |
| Category_Office Supplies   | 49.51    | <0.001  |
| Category_Technology        | 41.44    | <0.001  |
| Region_East                | -11.31   | 0.045   |
| Region_South               | -15.13   | 0.020   |
| Region_West                | -15.50   | 0.005   |

**折扣每增加 1 个百分点，利润平均下降 2.37 美元**（在控制其他变量下)。办公用品和技术类别的利润显著高于家具类别，东部和南部地区利润相对较低。

#### 3.5.2 亏损订单深度分析

![亏损产品分析](subcat_profit_avg_discount.png)

- **亏损订单总数**：1,871 笔（占比 18.72%）
- **亏损子类别中平均折扣最高的三个**：
  - Binders：37.2%
  - Machines：30.6%
  - Tables：26.1%
- **同一子类别（以 Tables 为例）内盈利订单与亏损订单的平均折扣对比**：
  - 盈利订单平均折扣：7.5%
  - 亏损订单平均折扣：36.5%

**结论**：即使是同一产品，亏损订单的折扣也远高于盈利订单，直接证明过度折扣是亏损主因。

### 3.6  预测性分析：随机森林模型

使用随机森林分类器预测订单是否亏损（标签 is_loss），特征包括：Discount、Sales、Quantity、Region、Category、Sub-Category、is_holiday_season。模型表现：

- 准确率：95%
- 召回率（亏损类）：84%
- 特征重要性（Top 5）：
    - Discount：56.8%
    - Sales：19.2%
    - Quantity：5.7%
    - Sub-Category_Binders：2.8%
    - Sub-Category_Tables：2.0%

**折扣是预测亏损的最重要特征，重要性占比超过 56%。** 部分依赖图显示，当折扣超过 20% 时，亏损概率急剧上升。

### 3.7 客户聚类分析（KMeans）

基于客户 RFM 指标（recency, frequency, monetary, avg_discount）进行 KMeans 聚类（k=4），结果如下：

| 簇 | 平均折扣 | 平均利润 | 特征描述 |
|---|--------|--------|----------|
| 0 | 0.26   | -12.19 | 高折扣亏损型 – 利润为负，折扣高 |
| 1 | 0.15   | 726.97 | 高价值活跃型 – 利润最高，购买频率高（8.95），近期购买（82天） |
| 2 | 0.15   | 195.26 | 沉睡普通型 – 近 1.6 年未购买，频率低（3.8） |
| 3 | 0.09   | 328.68 | 低折扣稳健型 – 利润良好，折扣敏感度低 |

**策略启示：**
- 簇0：立即收紧折扣政策，甚至考虑淘汰。
- 簇1：重点维护，保持折扣水平，提升复购。
- 簇2：小成本唤醒活动。
- 簇3：维持低折扣策略，增加产品曝光。

### 3.8 折扣上线模拟(Excel)

基于亏损子类（Tables、Bookcases）构建折扣上限模拟器。将折扣上限设为 20% 时，可减少亏损约 $82,196。推荐将折扣上限设置为 20% 以平衡促销效果与利润。

## 四、核心结论

- 折扣是亏损的首要因素：亏损订单的平均折扣（48%）远高于盈利订单（8%），线性回归和随机森林均证实折扣的负向影响最大。
- 产品层面：Tables、Bookcases 是主要亏损源，且折扣过高是直接原因。
- 客户层面：存在“高折扣亏损型”客户群（簇0），应调整策略。
- 区域层面：中部地区利润最低，需重点优化。

## 五、业务建议

| 建议         | 具体措施       | 预期效果    |
|--------------|------------------------|----------------------|
| 限制高折扣   | 对 Tables、Bookcases 设置 20% 的折扣上限；对折扣超过 30% 的订单进行人工审核 | 减少亏损订单约 37%，提升利润 $80,000+       |
| 差异化促销   | 对高折扣亏损客群（簇0）停止统一高折扣，改用满减或捆绑销售                 | 降低无效折扣成本，改善客户质量             |
| 区域优化     | 在中部地区开展满减活动替代直接打折，降低物流成本                         | 提升区域利润                               |
| 产品组合     | 将高利润产品（如 Copiers）与低利润产品（如 Tables）捆绑销售              | 提升客单价，改善低利润产品盈利             |
| 实时监控     | 部署随机森林模型于订单系统，对预测亏损概率 > 0.7 的订单预警              | 提前干预，减少亏损                         |

## 六、项目成果与文件清单

- Python Notebook：sample-superstore-analysis.ipynb（包含数据清洗、EDA、回归、随机森林、聚类等完整代码）
- SQL 脚本：生成 RFM 客户分层表、orders_wide 宽表
- Excel 模拟器：discount_simulator.xlsx（动态设置折扣上限，实时计算利润变化）
- Power BI 仪表板：交互式看板（散点图、树状图、地图、参数模拟）
- 报告文档：本文档

## 七、附录

### 7.1 数据透视表

![数据透视表](Pivot_Table.png)

### 7.2 模拟利润表（部分截图）

![模拟利润表](Profit_Simulation.png)

### 7.3 仪表盘截图

![仪表盘截图](Dashboard_Screenshot.png)